# Fashion Image Generator Using DCGAN
Mini application (lightweight: small model, 3,000-image subset, ~8 epochs): trains a DCGAN on a small subset of Fashion-MNIST and generates new clothing-like images.

Sections: Backend → Frontend → Test Cases

In [ ]:
# !pip install torch torchvision ipywidgets matplotlib
import os, copy, torch, torch.nn as nn
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

LATENT_DIM = 100
CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

## Backend

### Dataset loader (small subset)

In [ ]:
def synthetic_fashion(n):
    # Fallback if Fashion-MNIST cannot be downloaded: random shirt / trouser / shoe-like shapes
    X = torch.full((n, 1, 28, 28), -1.0)
    for i in range(n):
        t = np.random.randint(3); v = np.random.uniform(0.3, 1.0) * 2 - 1
        if t == 0:   X[i, 0, 6:24, 7:21] = v; X[i, 0, 6:12, 3:25] = v       # shirt
        elif t == 1: X[i, 0, 4:26, 8:13] = v; X[i, 0, 4:26, 15:20] = v      # trousers
        else:        X[i, 0, 16:23, 3:25] = v; X[i, 0, 11:16, 3:10] = v     # shoe
    return torch.utils.data.TensorDataset(X, torch.zeros(n))

def load_data(subset_size=3000, batch_size=128):
    try:
        tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])  # scale to [-1, 1]
        full = datasets.FashionMNIST("data", train=True, download=True, transform=tfm)
        ds = Subset(full, torch.randperm(len(full))[:subset_size].tolist())
    except Exception as e:
        print("Fashion-MNIST download failed, using synthetic data:", e)
        ds = synthetic_fashion(subset_size)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=0)

### Generator and Discriminator

In [ ]:
class Generator(nn.Module):
    def __init__(self, z=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z, 64, 7, 1, 0, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),   # 7x7
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),   # 14x14
            nn.ConvTranspose2d(32, 1, 4, 2, 1, bias=False), nn.Tanh())                              # 28x28
    def forward(self, x): return self.net(x.view(x.size(0), -1, 1, 1))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                         # 14x14
            nn.Conv2d(32, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),    # 7x7
            nn.Conv2d(64, 1, 7, 1, 0, bias=False))                                                  # logit
    def forward(self, x): return self.net(x).view(-1)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)): nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d): nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

### Training loop with loss record and checkpoint saving

In [ ]:
criterion = nn.BCEWithLogitsLoss()
fixed_noise = torch.randn(32, LATENT_DIM)

def train_step(G, D, optG, optD, real):
    b = real.size(0)
    real_lbl, fake_lbl = torch.ones(b), torch.zeros(b)
    # Discriminator: real -> 1, fake -> 0
    z = torch.randn(b, LATENT_DIM)
    fake = G(z)
    optD.zero_grad()
    d_loss = criterion(D(real), real_lbl) + criterion(D(fake.detach()), fake_lbl)
    d_loss.backward(); optD.step()
    # Generator: make D label fakes as real
    optG.zero_grad()
    g_loss = criterion(D(fake), real_lbl)
    g_loss.backward(); optG.step()
    return d_loss.item(), g_loss.item()

def train(epochs=8, subset_size=3000, batch_size=128, lr=2e-4, on_epoch=None):
    loader = load_data(subset_size, batch_size)
    G, D = Generator(), Discriminator()
    G.apply(init_weights); D.apply(init_weights)
    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    history = {"d_loss": [], "g_loss": [], "samples": []}
    for ep in range(1, epochs + 1):
        dl, gl = [], []
        for real, _ in loader:
            d, g = train_step(G, D, optG, optD, real)
            dl.append(d); gl.append(g)
        history["d_loss"].append(float(np.mean(dl))); history["g_loss"].append(float(np.mean(gl)))
        G.eval()
        with torch.no_grad(): history["samples"].append(G(fixed_noise).cpu())
        G.train()
        torch.save({"epoch": ep, "G": G.state_dict(), "D": D.state_dict(),
                    "d_loss": history["d_loss"], "g_loss": history["g_loss"]}, f"{CKPT_DIR}/dcgan_epoch_{ep}.pt")
        if on_epoch: on_epoch(ep, epochs, history)
    return G, D, history

def generate(G, n=16):
    G.eval()
    with torch.no_grad(): return G(torch.randn(n, LATENT_DIM)).cpu()

## Frontend

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

def show_grid(imgs, title, nrow=8):
    g = make_grid(imgs, nrow=nrow, normalize=True, value_range=(-1, 1)).permute(1, 2, 0)
    plt.figure(figsize=(7, 7)); plt.imshow(g.numpy(), cmap="gray"); plt.axis("off"); plt.title(title); plt.show()

epochs_w = w.IntSlider(value=8, min=1, max=20, description="Epochs")
subset_w = w.IntSlider(value=3000, min=1000, max=10000, step=1000, description="Subset")
batch_w  = w.Dropdown(options=[64, 128, 256], value=128, description="Batch")
lr_w     = w.FloatLogSlider(value=2e-4, base=10, min=-5, max=-3, step=0.1, description="LR")
btn      = w.Button(description="Train", button_style="success")
status   = w.Label("Idle")
bar      = w.IntProgress(min=0, max=100, description="Progress")
gallery  = w.Output()
state = {}

def on_epoch(ep, total, h):
    bar.max, bar.value = total, ep
    status.value = f"Epoch {ep}/{total}  |  D loss {h['d_loss'][-1]:.3f}  |  G loss {h['g_loss'][-1]:.3f}"
    with gallery:
        clear_output(wait=True)
        show_grid(h["samples"][-1], f"Generated images - epoch {ep}")

def on_train(_):
    btn.disabled = True; gallery.clear_output()
    G, D, h = train(epochs_w.value, subset_w.value, batch_w.value, lr_w.value, on_epoch)
    state.update(G=G, D=D, history=h)
    status.value += "  |  Done"
    btn.disabled = False

btn.on_click(on_train)
display(w.VBox([epochs_w, subset_w, batch_w, lr_w, btn, bar, status, gallery]))

### Final generated images and loss record
Run after training finishes.

In [ ]:
G, h = state["G"], state["history"]
show_grid(generate(G, 32), "New synthetic clothing images")
plt.figure(figsize=(6, 3.5))
plt.plot(h["d_loss"], label="Discriminator"); plt.plot(h["g_loss"], label="Generator")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training losses"); plt.legend(); plt.show()

## Test Cases

In [ ]:
# 1. GAN architecture
G_t, D_t = Generator(), Discriminator()
z = torch.randn(8, LATENT_DIM)
out = G_t(z)
assert out.shape == (8, 1, 28, 28), out.shape
assert out.min() >= -1 and out.max() <= 1
assert D_t(out).shape == (8,)
print("Test 1 PASS: architecture (G: z->1x28x28, D: image->score)")

# 2. Adversarial training: one step must update both networks and give finite losses
G_t.apply(init_weights); D_t.apply(init_weights)
oG = torch.optim.Adam(G_t.parameters(), 2e-4, betas=(0.5, 0.999))
oD = torch.optim.Adam(D_t.parameters(), 2e-4, betas=(0.5, 0.999))
g0, d0 = copy.deepcopy(G_t.state_dict()), copy.deepcopy(D_t.state_dict())
dl, gl = train_step(G_t, D_t, oG, oD, torch.randn(16, 1, 28, 28).clamp(-1, 1))
changed = lambda a, b: any(not torch.equal(a[k], b[k]) for k in a if a[k].dtype.is_floating_point)
assert changed(g0, G_t.state_dict()) and changed(d0, D_t.state_dict()) and np.isfinite([dl, gl]).all()
print(f"Test 2 PASS: adversarial step updated G and D (D loss {dl:.3f}, G loss {gl:.3f})")

# 3. Checkpoints saved and loadable
ck = torch.load(f"{CKPT_DIR}/dcgan_epoch_{len(state['history']['d_loss'])}.pt")
G_t.load_state_dict(ck["G"]); D_t.load_state_dict(ck["D"])
print("Test 3 PASS: checkpoint saved and reloaded")

# 4. Training progress vs generated quality (pixel statistics gap to real data; lower = closer)
real_batch = next(iter(load_data(2000, 2000)))[0]
def gap(imgs): return abs(imgs.mean() - real_batch.mean()).item() + abs(imgs.std() - real_batch.std()).item()
s = state["history"]["samples"]
first, last = gap(s[0]), gap(s[-1])
print(f"Gap to real data: epoch 1 = {first:.3f}, final epoch = {last:.3f}")
print("Test 4", "PASS: quality improved with training" if last < first else "CHECK: train for more epochs")